<a href="https://colab.research.google.com/github/simecek/dspracticum2026/blob/main/lesson04/gradio_minimal.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Minimal Gradio apps

[Gradio](https://www.gradio.app) turns a Python **function** into a small **web page**: the user fills in the inputs, Gradio calls your function and shows what it returns.

In this notebook:
1. a text app: type your name, get a greeting
2. a picture app: upload a picture, get a bar chart. This is the same kind of output as your classifier in Assignment 02.
3. how to put the app online on Hugging Face Spaces

No GPU needed: the default CPU runtime is enough.

In [ ]:
try:
    import gradio as gr
except ImportError:
    !pip install -q gradio
    import gradio as gr

print(gr.__version__)

---
## 1. Text in, text out

Every Gradio app has three parts:
- `fn`: **your function**, which does the work
- `inputs`: what the user gives the function (here a text box)
- `outputs`: how the result is shown (here another text box)

Run the cell. The app appears right below it: type your name and click *Submit*.

In [ ]:
# app.py
import gradio as gr


def pozdrav(jmeno):
    """Vezme jméno a vrátí pozdrav."""
    return f"Ahoj {jmeno}!"


demo = gr.Interface(
    fn=pozdrav,                              # funkce, která se zavolá
    inputs=gr.Textbox(label="Tvoje jméno"),  # vstupní políčko
    outputs=gr.Textbox(label="Pozdrav"),     # výstupní políčko
    title="Ahoj aplikace",
    description="Napiš své jméno a aplikace tě pozdraví.",
)

demo.launch()

`pozdrav` is an ordinary Python function. You can call it without any app:

In [ ]:
pozdrav("Ada")

**Try it:** Add a second input, e.g. `gr.Slider(1, 100, label="Věk")`. Then `inputs` becomes a list, `inputs=[gr.Textbox(...), gr.Slider(...)]`, and the function gets two arguments: `def pozdrav(jmeno, vek):`.

In [ ]:
# your code here

---
## 2. Picture in, bar chart out

Now the input is a picture. Gradio gives your function the picture as a **NumPy array** of shape `(height, width, 3)`: for every pixel three numbers 0-255, the amount of red, green and blue.

Our function computes what share of the picture is red, green and blue. It returns a **dictionary** `{name: number}`, and `gr.Label` shows it as a bar chart.

In [ ]:
import numpy as np


def barvy(obrazek):
    """Vezme obrázek a spočítá, jaký podíl tvoří červená, zelená a modrá."""
    prumery = obrazek[:, :, :3].mean(axis=(0, 1))   # průměrná červená, zelená a modrá (0-255)
    podily = prumery / max(prumery.sum(), 1)        # podíly, které dají dohromady 1 (max: černý obrázek nedělí nulou)
    return {"červená": float(podily[0]), "zelená": float(podily[1]), "modrá": float(podily[2])}

Before we build the app, let's look inside. We make a tiny picture ourselves: 100 × 100 pixels, all of them pure red.

In [ ]:
cerveny = np.zeros((100, 100, 3), dtype=np.uint8)   # 100 x 100 pixels, everything 0 = black
cerveny[:, :, 0] = 255                              # channel 0 = red at maximum
print(cerveny.shape)
barvy(cerveny)

Now the app. The only changes from section 1 are the input (`gr.Image`) and the output (`gr.Label`). Upload any photo, or drag one from your computer:

In [ ]:
demo = gr.Interface(
    fn=barvy,
    inputs=gr.Image(label="Tvůj obrázek"),   # obrázek přijde do funkce jako NumPy pole
    outputs=gr.Label(label="Podíl barev"),   # slovník {název: číslo} se ukáže jako sloupce
    title="Barvy na obrázku",
    description="Nahraj obrázek a aplikace spočítá, kolik je na něm červené, zelené a modré.",
)

demo.launch()

**This is exactly the shape of your Assignment 02 app.** Replace the body of the function with your model's prediction, and return a dictionary *category → probability*:

```python
def classify_image(image):
    prediction, index, probabilities = learn.predict(image)
    return dict(zip(categories, map(float, probabilities)))
```

The whole app is in [lesson03/gradio_app/app.py](https://github.com/simecek/dspracticum2026/blob/main/lesson03/gradio_app/app.py).

**Try it:** Change `gr.Label()` to `gr.Label(num_top_classes=1)`. Or return the picture mirrored: `outputs=gr.Image()` and `return obrazek[:, ::-1]`.

In [ ]:
# your code here

---
## 3. Putting the app online

The app above only lives while this notebook runs. To get a permanent public link, put it on [Hugging Face Spaces](https://huggingface.co/spaces) (free):

1. Save the code into a file `app.py` (the next cell does that) and download it: folder icon on the left → right-click `app.py` → *Download*.
2. On Hugging Face: *New Space* → choose a name → SDK **Gradio** → hardware **CPU basic** (free) → *Create Space*.
3. *Files* → *Add file* → *Upload files* → upload `app.py` → *Commit*.
4. Wait a minute while the Space builds. Your app is now online.

These two examples need no `requirements.txt`, because Gradio and NumPy are already installed on a Gradio Space. Your Assignment 02 app needs one (fastai, timm, ...), see [lesson03/gradio_app](https://github.com/simecek/dspracticum2026/tree/main/lesson03/gradio_app).

In [ ]:
%%writefile app.py
import gradio as gr
import numpy as np


def barvy(obrazek):
    """Vezme obrázek a spočítá, jaký podíl tvoří červená, zelená a modrá."""
    prumery = obrazek[:, :, :3].mean(axis=(0, 1))   # průměrná červená, zelená a modrá (0-255)
    podily = prumery / max(prumery.sum(), 1)        # podíly, které dají dohromady 1 (max: černý obrázek nedělí nulou)
    return {"červená": float(podily[0]), "zelená": float(podily[1]), "modrá": float(podily[2])}


demo = gr.Interface(
    fn=barvy,
    inputs=gr.Image(label="Tvůj obrázek"),
    outputs=gr.Label(label="Podíl barev"),
    title="Barvy na obrázku",
    description="Nahraj obrázek a aplikace spočítá, kolik je na něm červené, zelené a modré.",
)

demo.launch()